# Logistic Regression and Model Evaluation – Concepts

**Slides:** Part 04 «Classification models» – logistic regression, evaluation (confusion matrix and metrics,
classification thresholds, ROC curve and AUC).

This notebook summarises the concepts of the slides and adds explanations and small examples. All examples use the numbers of the slides or small simulated data – no data set is loaded.

**Contents**
1. Logistic regression: a linear model for classification
2. Fitting by maximum likelihood (log-loss)
3. Confusion matrix and metrics
4. Classification thresholds
5. ROC curve and AUC

## Libraries and settings

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Logistic regression: a linear model for classification

$$P(y = 1 \mid x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

In words: a linear **score** $\beta_0 + \beta_1 x$ is squeezed by an S-curve (the *sigmoid* $\sigma$) into a probability
between 0 and 1. With several features the score is $\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$.

- $\beta_1 > 0$: the probability rises with $x$.
- Example: $\beta_0 = -5$, $\beta_1 = 1$, $x = 6$ → score 1 → $p = 1/(1+e^{-1}) \approx 0.73$ → class 1 (if $\tau = 0.5$).

**Additional explanation – odds.** $\log\frac{p}{1-p} = \beta_0 + \beta_1 x$: the score is the log-odds. $e^{\beta_1}$ is
the factor by which the odds $p/(1-p)$ change when $x$ increases by one unit.

In [ ]:
sigmoid = lambda s: 1 / (1 + np.exp(-s))
print('beta_0 = -5, beta_1 = 1, x = 6 -> p =', round(sigmoid(-5 + 1 * 6), 3))

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 200)
y = (rng.uniform(size=200) < sigmoid(-5 + 1 * x)).astype(int)
model = LogisticRegression().fit(x.reshape(-1, 1), y)
grid = np.linspace(0, 10, 300).reshape(-1, 1)

plt.figure(figsize=(7, 4))
plt.scatter(x, y + rng.normal(0, 0.02, 200), s=12, alpha=0.6, color='black')
plt.plot(grid, model.predict_proba(grid)[:, 1], linewidth=2, label='$\\hat p(x) = \\sigma(\\hat\\beta_0 + \\hat\\beta_1 x)$')
plt.axhline(0.5, color='tab:orange', linestyle='--', label='threshold τ = 0.5')
plt.xlabel('x')
plt.ylabel('P(y = 1 | x)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('Estimated coefficients:', round(model.intercept_[0], 2), round(model.coef_[0][0], 2))

## 2. Fitting by maximum likelihood (log-loss)

Logistic regression is fitted by **maximum likelihood**: choose $\beta$ so that the observed classes are as plausible as
possible. Maximising the likelihood is the same as minimising the average **log-loss** (cross-entropy):

$$\text{log-loss} = -\frac{1}{n}\sum_{i=1}^n \Big[y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i)\Big]$$

There is no closed-form solution – the coefficients are computed numerically (e.g. by gradient-based methods). A
confident wrong prediction ($\hat p = 0.99$ for a negative case) is penalised heavily.

In [ ]:
for p_hat in [0.9, 0.6, 0.5, 0.1, 0.01]:
    print(f'y = 1, p_hat = {p_hat:4}: loss = {-np.log(p_hat):.3f}')

## 3. Confusion matrix and metrics

| | predicted negative | predicted positive |
|---|---|---|
| **actual negative** | TN | FP |
| **actual positive** | FN | TP |

This is the orientation of scikit-learn: `confusion_matrix(y_true, y_pred)` returns `[[TN, FP], [FN, TP]]` (rows =
actual class, columns = predicted class, ordered 0, 1).

| metric | formula | question |
|---|---|---|
| accuracy | $\frac{TP + TN}{n}$ | share of all predictions that are correct |
| precision | $\frac{TP}{TP + FP}$ | of all predicted positives, how many are really positive? |
| recall (TPR, sensitivity) | $\frac{TP}{TP + FN}$ | of all real positives, how many did we find? |
| FPR | $\frac{FP}{FP + TN}$ | of all real negatives, how many were wrongly flagged? |
| F1 | $2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$ | one number balancing precision and recall |

**Caution:** with imbalanced classes, accuracy alone misleads. Slide example with $n = 171$ test observations:

In [ ]:
TP, FP, FN, TN = 50, 22, 24, 75
n = TP + FP + FN + TN
precision, recall = TP / (TP + FP), TP / (TP + FN)
print(f'n = {n}')
print(f'accuracy  = {(TP + TN) / n:.3f}')
print(f'precision = {precision:.3f}')
print(f'recall    = {recall:.3f}')
print(f'FPR       = {FP / (FP + TN):.3f}')
print(f'F1        = {2 * precision * recall / (precision + recall):.3f}')

# Why accuracy misleads with imbalanced classes: 1000 cases, 20 positives, model predicts always "negative"
print('\nAlways "negative" with 2 % positives: accuracy =', 980 / 1000, ', recall = 0')

## 4. Classification thresholds

$$\hat y = 1 \quad \text{if} \quad \hat p(x) \ge \tau$$

- $\tau$ ↑: fewer predicted positives – recall and FPR cannot increase; precision often rises, but this is not
  guaranteed.
- $\tau$ ↓: more predicted positives – recall and FPR cannot decrease.
- The default $\tau = 0.5$ is common for probability outputs; score-based classifiers (e.g. SVM) use an analogous
  threshold.

**Rule:** pick $\tau$ from the **costs** of false positives vs. false negatives. If a false negative costs $c$ times as
much as a false positive, the cost-optimal threshold of a well-calibrated model is about $\tau^* = 1/(1+c)$.

In [ ]:
rng = np.random.default_rng(1)
y_true = np.r_[np.zeros(600), np.ones(400)]
p_hat = np.r_[rng.beta(3, 5, 600), rng.beta(6, 3, 400)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(p_hat[y_true == 0], bins=30, alpha=0.7, label='actual negatives (y = 0)')
axes[0].hist(p_hat[y_true == 1], bins=30, alpha=0.7, label='actual positives (y = 1)')
axes[0].axvline(0.5, color='black', linewidth=2, label='threshold τ = 0.5')
axes[0].set_xlabel('predicted score $\\hat p(x)$')
axes[0].set_ylabel('count')
axes[0].set_title('FN: positives left of τ, FP: negatives right of τ')
axes[0].legend()

taus = np.linspace(0.05, 0.95, 19)
rec = [np.mean(p_hat[y_true == 1] >= t) for t in taus]
fpr = [np.mean(p_hat[y_true == 0] >= t) for t in taus]
prec = [np.sum((p_hat >= t) & (y_true == 1)) / max(np.sum(p_hat >= t), 1) for t in taus]
axes[1].plot(taus, rec, marker='o', label='recall')
axes[1].plot(taus, prec, marker='o', label='precision')
axes[1].plot(taus, fpr, marker='o', label='FPR')
axes[1].set_xlabel('threshold τ')
axes[1].set_title('Metrics as a function of the threshold')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

## 5. ROC curve and AUC

**ROC curve:** for every threshold $\tau$ plot (FPR, TPR) and connect the points. Each point is one threshold; the
diagonal corresponds to a random ranking (in expectation).

**AUC** = area under the ROC curve (0.5 = random ranking, 1 = perfect). Interpretation: the **probability that a random
positive case gets a higher score than a random negative case** (ties count ½). Example: AUC = 0.95 → in 95 % of such
pairs the positive case is ranked higher.

**Additional explanation.** The AUC measures the *ranking* quality of a model and does not depend on the threshold. A
model with a high AUC can still give bad results with an unsuitable threshold – and the AUC says nothing about whether the
probabilities are well calibrated.

In [ ]:
fpr_c, tpr_c, thr = roc_curve(y_true, p_hat)
auc = roc_auc_score(y_true, p_hat)

plt.figure(figsize=(5.5, 5.5))
plt.plot(fpr_c, tpr_c, linewidth=2, label=f'model (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='random (AUC = 0.5)')
for t in [0.3, 0.5, 0.7]:
    plt.scatter(np.mean(p_hat[y_true == 0] >= t), np.mean(p_hat[y_true == 1] >= t), s=60, zorder=3)
    plt.annotate(f'τ = {t}', (np.mean(p_hat[y_true == 0] >= t), np.mean(p_hat[y_true == 1] >= t)),
                 xytext=(8, -12), textcoords='offset points')
plt.xlabel('false positive rate (FPR)')
plt.ylabel('true positive rate (TPR)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

# AUC as the probability of a correct ranking (all positive-negative pairs)
pos, neg = p_hat[y_true == 1], p_hat[y_true == 0]
pairs = (pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean()
print(f'AUC = {auc:.4f}, share of correctly ranked pairs = {pairs:.4f}')

**Key messages**
- Logistic regression turns a linear score into a probability; it is fitted by maximum likelihood (log-loss).
- With imbalanced classes look at precision, recall and FPR – not only at the accuracy.
- The threshold is a business decision based on the costs of errors.
- The ROC curve shows all thresholds at once; the AUC measures how well the model ranks positives above negatives.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')